# Question 2 — Does morphological diversity change over geological time?

**Approach**: since we don't have species labels for individual fossil specimens, we use unsupervised clustering as a proxy for morphological diversity within each time-slice sample. More distinct clusters in a sample = more morphologically distinct sub-groups present = a proxy for higher species richness at that point in geological time.

**Pipeline**:
1. Run KMeans on each of the ~370 individual sample files independently, selecting cluster count via the elbow method
2. Map each sample to its geological age using the shared mastersheet
3. Test whether cluster count (diversity proxy) trends over time using moving averages, LOESS regression, and Pearson/Spearman correlation
4. Compare an early (2.5-5 Ma) vs. late (0-2.5 Ma) period directly

## Setup

In [ ]:
from pathlib import Path

# All paths are relative to this notebook's location (notebooks/), so the repo can be
# cloned and run anywhere without editing paths.
DATA_DIR = Path("../data")


In [ ]:
import os
import multiprocessing
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import re
from tqdm import tqdm
from sklearn.cluster import KMeans
import statsmodels.api as sm
import seaborn as sns
import warnings

In [ ]:
# Suppress memory leak warning (optional)
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# Set OMP_NUM_THREADS dynamically to avoid KMeans memory leak issue on Windows
num_threads = max(1, multiprocessing.cpu_count() - 1)  # Use all but one core
os.environ["OMP_NUM_THREADS"] = str(num_threads)

## Stage 1 — Per-sample KMeans clustering

Each of the ~370 raw fossil measurement files represents one geological time-slice sample. For each one, we standardise the selected morphometric features and find the optimal number of KMeans clusters via the elbow method (WCSS across k = 2-59).

**Note on reproducibility**: the raw per-sample `.xlsx` files (`Final Ceara Rise Data/`) are not included in this repo due to size — see the README. This stage won't run without them, but its output (`selected_features_clustering_summary.xlsx`) is included, so Stage 2 onward runs standalone.

In [ ]:
folder_path = DATA_DIR / "Final Ceara Rise Data"

In [ ]:
summary_folder_path = DATA_DIR

In [ ]:
os.makedirs(summary_folder_path, exist_ok=True)

In [ ]:
summary_data = []

In [ ]:
# Define the features to use for clustering (Modify this list for future use)
selected_features = [
    "area", "sphericity", "shapefactor", 
    "min(diameter)", "max(diameter)", "mean(diameter)"
]

In [ ]:
def load_and_preprocess(file_path):
    # Read the first sheet of the Excel file
    df = pd.read_excel(file_path, sheet_name=0)
    
    # Standardize column names
    df.columns = [col.strip().replace(".", "").replace(" ", "").lower() for col in df.columns]
    
    # Ensure selected features exist in the file
    available_features = [col for col in df.columns if col in selected_features]
    
    if not available_features:
        print(f"Skipping {file_path} (No valid selected features found)")
        return None
    
    # Keep only the selected features and drop NaNs
    df = df[available_features].dropna()
    
    # Normalize the data
    scaler = StandardScaler()
    df_scaled = scaler.fit_transform(df)
    
    return df_scaled


In [ ]:
def find_optimal_k(data_scaled, file_name):
    distortions = []
    k_range = range(2, 60)  # Start from 2 since K=1 is trivial

    # Compute WCSS for each k
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(data_scaled)
        distortions.append(kmeans.inertia_)

    # Compute second derivative (curvature)
    distortions_diff2 = np.diff(distortions, 2)  # Equivalent to np.diff(np.diff(distortions))

    # Find the elbow as the point where curvature is the lowest (most negative)
    elbow_point = np.argmin(distortions_diff2) + 2  # Adjust index shift

    return elbow_point


In [ ]:
# Process all files in the folder
for file_name in tqdm(os.listdir(folder_path)):
    if file_name.endswith(".xlsx"):
        file_path = os.path.join(folder_path, file_name)
        df_scaled = load_and_preprocess(file_path)

        if df_scaled is not None:
            # Determine optimal k using Elbow Point Method
            optimal_k = find_optimal_k(df_scaled, file_name)

            # Store result in summary data
            summary_data.append({"File Name": os.path.basename(file_path), "Number of Clusters": optimal_k})

In [ ]:
# Save summary results to an Excel file
summary_df = pd.DataFrame(summary_data)
summary_file = os.path.join(summary_folder_path, "selected_features_clustering_summary.xlsx")
summary_df.to_excel(summary_file, index=False)
print(f"Summary saved: {summary_file}")

## Stage 2 — Map geological age onto each sample's cluster count

The clustering summary only has filenames. We extract each file's site/hole/core/section metadata from its filename and look up the matching geological age (Ma) in the mastersheet.

In [ ]:
file_path = DATA_DIR / "selected_features_clustering_summary.xlsx"
df = pd.read_excel(file_path)

In [ ]:
# Loading the Filtered Master Sheet File
mastersheet_file_path = DATA_DIR / "Mastersheet.xlsx"
df_master = pd.read_excel(mastersheet_file_path, sheet_name="Sheet1", header=0)
df_master = df_master[['Age (Ma)', 'SITE', 'HOLE', 'CORE', 'SECTION', 'TOP_DEPTH', 'BOTTOM_DEPTH']]

In [ ]:
# Now, Mapping Age (Ma) to Each Fossil File Name in the Summary
def get_age_from_filename(file_name):
    metadata = extract_metadata(file_name)
    if metadata is None:
        print(f"Extraction failed for  '{file_name}'.")
        return None
    matching = df_master[
        (df_master['SITE'] == metadata['SITE']) &
        (df_master['HOLE'] == metadata['HOLE']) &
        (df_master['CORE'] == metadata['CORE']) &
        (df_master['SECTION'] == metadata['SECTION']) &
        (df_master['TOP_DEPTH'] <= metadata['TOP_DEPTH']) &
        (df_master['BOTTOM_DEPTH'] >= metadata['BOTTOM_DEPTH'])
    ]
    if matching.empty:
        print(f"No matching mastersheet row found for file '{file_name}'.")
        return None
    else:
        return matching.iloc[0]['Age (Ma)']

In [ ]:
# Apply the function to add a new "Age (Ma)" column to the clustering summary.
if 'File Name' not in df.columns:
    print("Error")
else:
    df['Age (Ma)'] = df['File Name'].apply(get_age_from_filename)

In [ ]:
output_file = DATA_DIR / "selected_features_clustering_summary_with_age.xlsx"
df.to_excel(output_file, index=False)
print(f"Updated clustering summary file with mapped Age (Ma) has been saved as '{output_file}'.")

## Stage 3 — Does diversity (cluster count) trend with geological age?

We test this three ways: raw correlation (Pearson + Spearman), a smoothed trend line (moving average and LOESS), and a direct early-vs-late period comparison.

In [ ]:
file_path = DATA_DIR / "selected_features_clustering_summary_with_age.xlsx"
df = pd.read_excel(file_path, sheet_name="Sheet1")

In [ ]:
# Extract relevant columns
age = df["Age (Ma)"]
num_clusters = df["Number of Clusters"]

In [ ]:
# Compute Pearson and Spearman correlation
pearson_corr, pearson_pval = pearsonr(age, num_clusters)
spearman_corr, spearman_pval = spearmanr(age, num_clusters)

In [ ]:
# Display results
print(f"Pearson Correlation: {pearson_corr:.4f}, p-value: {pearson_pval:.4f}")
print(f"Spearman Correlation: {spearman_corr:.4f}, p-value: {spearman_pval:.4f}")

In [ ]:
# Sort data by Age (Ma) for proper visualization
df_sorted = df.sort_values(by="Age (Ma)")

In [ ]:
# Plot age vs clusters with a connecting line
plt.figure(figsize=(10, 5))
plt.plot(df_sorted["Age (Ma)"], df_sorted["Number of Clusters"], marker="o", linestyle="-", color="b", label="Data Points")
plt.xlabel("Age (Ma)")
plt.ylabel("Number of Clusters (Species)")
plt.title("Number of Species Clusters vs. Age Over Geological Time")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Define a moving average function
def moving_average(data, window_size):
    return data.rolling(window=window_size, center=True).mean()

In [ ]:
# Apply a moving average with a window size of 5 (adjust if needed)
window_size = 5  # You can experiment with different values
df_sorted["Smoothed Clusters"] = moving_average(df_sorted["Number of Clusters"], window_size)

In [ ]:
# Plot the original and smoothed data
plt.figure(figsize=(10, 5))
plt.scatter(df_sorted["Age (Ma)"], df_sorted["Number of Clusters"], alpha=0.5, label="Original Data")
plt.plot(df_sorted["Age (Ma)"], df_sorted["Smoothed Clusters"], color="red", linewidth=2, label=f"Moving Average (window={window_size})")

# Labels and title
plt.xlabel("Age (Ma)")
plt.ylabel("Number of Clusters")
plt.title("Trend of Clusters Over Time (Smoothed)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Apply LOESS smoothing
lowess = sm.nonparametric.lowess(df_sorted["Number of Clusters"], df_sorted["Age (Ma)"], frac=0.1)  # Adjust frac for more/less smoothing

In [ ]:
# Extract smoothed values
smoothed_age = lowess[:, 0]
smoothed_clusters = lowess[:, 1]

In [ ]:
# Plot original data + LOESS smoothed trend
plt.figure(figsize=(10, 5))
plt.scatter(df_sorted["Age (Ma)"], df_sorted["Number of Clusters"], alpha=0.5, label="Original Data")
plt.plot(smoothed_age, smoothed_clusters, color="green", linewidth=2, label="LOESS Smoothed Trend")

# Labels and title
plt.xlabel("Age (Ma)")
plt.ylabel("Number of Clusters")
plt.title("LOESS Smoothed Trend of Clusters Over Time")
plt.legend()
plt.grid(True)
plt.show()

## Stage 4 — Early (2.5-5 Ma) vs. late (0-2.5 Ma) period comparison

**Finding**: cluster counts (our diversity proxy) show a mild declining trend in the more recent 2.5 million years, visible in both the distribution comparison and the smoothed trend above.

In [ ]:
# Define time periods for comparison
early_period = df_sorted[df_sorted["Age (Ma)"] <= 2.5]["Number of Clusters"]
late_period = df_sorted[df_sorted["Age (Ma)"] > 2.5]["Number of Clusters"]
print(f"Number of samples in Early Period (0-2.5 Ma): {len(early_period)}")
print(f"Number of samples in Late Period (2.5-5 Ma): {len(late_period)}")

In [ ]:
# Plot histograms for comparison
plt.figure(figsize=(10, 5))
sns.histplot(early_period, bins=20, kde=True, color="blue", label="Early Period (0-2.5 Ma)", alpha=0.5)
sns.histplot(late_period, bins=20, kde=True, color="red", label="Late Period (2.5-5 Ma)", alpha=0.5)
plt.xlabel("Number of Clusters")
plt.ylabel("Frequency")
plt.title("Comparison of Cluster Counts in Early vs. Late Periods")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
sns.boxplot(data=[early_period.to_numpy(), late_period.to_numpy()], palette=["blue", "red"])
plt.xticks([0, 1], ["Early Period (0-2.5 Ma)", "Late Period (2.5-5 Ma)"])
plt.ylabel("Number of Clusters")
plt.title("Boxplot Comparison of Cluster Counts")
plt.grid(True)
plt.show()